# 🎓 Mocogi MCP Client Demo

Dieses Notebook demonstriert die Nutzung des **Mocogi MCP Clients**, um Informationen über Studiengänge und Module der TH Köln abzufragen.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dgaida/modul_anerkennung/blob/master/notebooks/mcp_client_demo.ipynb)

## 🛠️ Installation und Setup

In [ ]:
!pip install git+https://github.com/dgaida/modul_anerkennung.git
!pip install fastmcp httpx llm-client gradio

## 🔑 Konfiguration

Hier kannst du deinen API-Key für das LLM und (optional) den Mocogi API Token setzen.

In [ ]:
import os
from google.colab import userdata

try:
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    # os.environ["MOCOGI_API_TOKEN"] = userdata.get("MOCOGI_API_TOKEN")
except Exception:
    print("Bitte stelle sicher, dass OPENAI_API_KEY in den Colab Secrets gesetzt ist.")
    os.environ["GROQ_API_KEY"] = "DEIN_API_KEY"

## 🖥️ Gradio Interface

Wir nutzen `llm_client` und verbinden uns mit dem MCP Server (hier simuliert durch direkten Import der Tools für die Demo-Zwecke im Notebook).

In [ ]:
import gradio as gr
from llm_client import LLMClient
from modul_anerkennung.mcp_client import MocogiClient

client = LLMClient(api_choice="gemini", llm="gemini-3.1-flash-lite-preview")
mcp_client = MocogiClient()

async def ask_mcp(question):
    async with mcp_client as mcp:
        # In einer echten MCP-Integration würde das LLM entscheiden, welches Tool es nutzt.
        # Hier nutzen wir den MocogiClient für den Datenabruf:
        if "studiengang" in question.lower() or "studiengänge" in question.lower():
            data = await mcp.list_study_programs()
            context = str(data)[:150000] # Gekürzt für das LLM
        elif "modul" in question.lower():
            # Standardmäßig MI Bachelor PO-5 für die Demo
            data = await mcp.get_modules_by_po("inf_mi5")
            context = str(data)[:150000]
        else:
            context = "Keine spezifischen Daten gefunden."

    prompt = f"""Nutze die folgenden Daten der TH Köln API, um die Frage des Nutzers zu beantworten:

    Daten:
    {context}

    Frage: {question}
    """

    response = client.chat_completion([{"role": "user", "content": prompt}])
    return response

iface = gr.Interface(
    fn=ask_mcp,
    inputs="text",
    outputs="text",
    title="Mocogi MCP Client Assistant",
    description="Frage nach Studiengängen oder Modulen der TH Köln."
)

iface.launch(debug=True, share=True)